# 02 · Preparação dos dados (T1)

**Onde estamos no ML:** esta é a etapa de *preparação* do entregável 2. Transforma o corpus bruto em exemplos supervisionados e justifica cada decisão. O código roda em `python -m textgloss.prepare` (e, na fase 7, dentro do DAG `preparacao_t1` do Airflow); o notebook mostra o resultado e as justificativas.

| Etapa | Tipo |
| --- | --- |
| Limpeza, normalização da notação (`slug()`, `gloss_keys`) | regra |
| spaCy: lema, UPOS, morfologia | extração de atributos |
| Rotulagem `MANTER`/`DESCARTAR` por alinhamento de lema | regra que gera os rótulos |
| Divisão treino/teste agrupada por frase | regra |
| **Classificador que decide** | **ML nosso (fases 3 a 6)** |

In [1]:
import json, re, sys
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from textgloss import config, corpus
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
palavras = pd.read_parquet(config.PROCESSED_DIR / "palavras.parquet")
frases = pd.read_parquet(config.PROCESSED_DIR / "frases.parquet")
palavras["rotulo_nome"] = palavras["rotulo"].map({1: "MANTER", 0: "DESCARTAR"})

## 1. Limpeza
Remove frases vazias, duplicatas exatas (texto + glosa) e glosas sem nenhum sinal (só marcas de pontuação).

In [2]:
rel = json.loads((config.PROCESSED_DIR / "relatorio_preparacao.json").read_text(encoding="utf-8"))
pd.Series({k: v for k, v in rel.items() if k.startswith(("linhas", "apos"))}, name="frases")

linhas_entrada                  127349
apos_remover_vazias             127349
apos_remover_duplicadas         127320
apos_remover_glosa_sem_sinal    127320
Name: frases, dtype: int64

**Justificativa.** Duplicatas inflariam o teste (a mesma frase em treino e teste). Glosas vazias não têm rótulo possível. Textos iguais com glosas diferentes são mantidos: são variação legítima entre intérpretes.

## 2. Normalização da notação
A glosa segue a notação do relatório técnico do VLibrasBD. Para comparar palavra com sinal, tudo vira `slug()` (minúsculas, sem acento) e a glosa é reduzida a um conjunto de chaves:

| Notação | Tratamento |
| --- | --- |
| `[PONTO]`, `[INTERROGAÇÃO]`, `[EXCLAMAÇÃO]`, `(+)`, `(-)` | removidas |
| `NÃO_GOSTAR` | `NÃO` e `GOSTAR` |
| `1S_AJUDAR_2S` | `AJUDAR` |
| `BANCO&DINHEIRO` | `BANCO` |
| `CADASTRO_NACIONAL` | inteiro e por partes |

In [3]:
exemplo = frases.sample(5, random_state=1)[["texto", "glosa"]].copy()
exemplo["chaves"] = exemplo["glosa"].map(lambda g: sorted(corpus.gloss_keys(g)))
pd.set_option("display.max_colwidth", 90); exemplo

,texto,glosa,chaves
63386,"Um total de 310 sacerdotes poloneses, sete alemães, seis tchecos, quatro luxemburguese...",TOTAL&RESULTADO 310 SACERDOTE POLÔNIA&PAÍS 7 ALEMANHA&PAÍS 6 CHECOSLOVÁQUIA&PAÍS 4 LUX...,"[2, 3, 310, 4, 6, 7, alemanha, belgica, checoslovaquia, holanda, luxemburgo, morto, po..."
13582,O interessado deverá enviar a solicitação junto com a documentação requerida por meio ...,INTERESSADO DEVER&OBRIGAÇÃO ENVIAR SOLICITAÇÃO JUNTO DOCUMENTAÇÃO REQUERER MEIO&MÉTODO...,"[anac, dever, documentacao, eletronica, enviar, interessado, junto, meio, peticionamen..."
114747,O Tom e a Mary foram lá fora.,TOM MARY FORA&SAIR [PONTO],"[fora, mary, tom]"
36484,Deve ser verificado em edital.,EU VERIFICAR EDITAL [PONTO],"[edital, eu, verificar]"
55933,Não sei onde ele mora.,NÃO_SABER ONDE ELE MORAR [PONTO],"[ele, morar, nao, onde, saber]"


## 3. Rotulagem T1
`MANTER` se o lema **ou** a forma escrita da palavra está entre as chaves da glosa; senão `DESCARTAR`. Usar o lema é essencial porque as glosas vêm no infinitivo (`interessado` → `INTERESSAR`).

**Duas correções que a primeira EDA revelou** (a rotulagem inicial gerava falsos `DESCARTAR`):

- **Clíticos e contrações:** o spaCy devolve `candidatar se` para `candidatar-se` e `em o` para `no`; passamos a aceitar também a **primeira parte** do lema, que é o que o intérprete glosa.
- **Numerais:** o intérprete escreve `7`, não `sete`; um dicionário de numerais por extenso → dígito evita marcar `sete` como descartado.

Esses ajustes são heurísticas nossas e entram na taxa de ruído medida na seção 8.

In [4]:
x = palavras[palavras.frase_id == frases.frase_id.iloc[10]]
print(frases.texto.iloc[10]); print(frases.glosa.iloc[10]); x[["palavra", "lema", "upos", "rotulo_nome"]]

Qualquer cidadão com nível superior completo (para candidatar-se ao Mestrado Acadêmico) ou com Mestrado completo (para candidatar-se ao Doutorado
QUALQUER CIDADÃO NÍVEL SUPERIOR COMPLETO CANDIDATAR MESTRADO ACADÊMICO MESTRADO COMPLETO CANDIDATAR DOUTORADO


,palavra,lema,upos,rotulo_nome
109,Qualquer,qualquer,DET,MANTER
110,cidadão,cidadão,NOUN,MANTER
111,com,com,ADP,DESCARTAR
112,nível,nível,NOUN,MANTER
113,superior,alto,ADJ,MANTER
114,completo,completo,ADJ,MANTER
115,para,para,SCONJ,DESCARTAR
116,candidatar-se,candidatar se,VERB,MANTER
117,ao,a o,ADP,DESCARTAR
118,Mestrado,mestrado,PROPN,MANTER


## 4. Atributos
Por palavra: classe gramatical, morfologia, relação de dependência, stopword, posição, tamanho e tipo da frase, classes gramaticais vizinhas e o lema.

In [5]:
atributos = [c for c in palavras.columns if c not in ("frase_id", "rotulo", "rotulo_nome", "split", "governo", "palavra")]
print(f"{len(atributos)} atributos:", ", ".join(atributos))

18 atributos: pos, lema, upos, dep, stop, pos_relativa, tam_frase, tipo_frase, upos_anterior, upos_seguinte, m_Gender, m_Number, m_Tense, m_VerbForm, m_Mood, m_Person, m_Polarity, m_PronType


**Decisões de encoding (aplicadas no treino, fase 3, dentro de um `Pipeline` do scikit-learn):**

- **Categóricos** (UPOS, dependência, morfologia, vizinhos, tipo de frase): one-hot. São poucas categorias e a regressão logística interpreta bem os coeficientes.
- **Lema:** one-hot só dos ~2.000 lemas mais frequentes; o resto vira `outro`. Sem isso a matriz teria dezenas de milhares de colunas e o modelo decoraria palavras raras.
- **Numéricos** (`pos_relativa`, `tam_frase`): padronizados para a regressão logística e o KNN; árvores não precisam.
- O encoder é ajustado **só no treino**, para não vazar informação do teste.

## 5. Valores ausentes

In [6]:
ausentes = (palavras[atributos].isin(["ausente", ""]).mean() * 100).round(1).sort_values(ascending=False)
print("% de 'ausente' por atributo (morfologia que não se aplica à palavra):"); print(ausentes.head(10).to_string())
print("\nlemas vazios:", int((palavras.lema == "").sum()), "· UPOS vazio:", int((palavras.upos == "").sum()))

% de 'ausente' por atributo (morfologia que não se aplica à palavra):
m_Polarity    98.9
m_Tense       91.8
m_Mood        91.5
m_Person      89.0
m_VerbForm    84.6
m_PronType    78.5
m_Gender      40.3
m_Number      26.4
lema           0.0
pos            0.0

lemas vazios: 0 · UPOS vazio: 0


**Decisão.** O spaCy devolve morfologia só quando ela se aplica (um substantivo não tem tempo verbal). Em vez de imputar, tratamos "não se aplica" como **categoria própria** (`ausente`), que é informação: diz que a palavra não é verbo.

## 6. Desbalanceamento

In [7]:
print(f"proporção MANTER: {rel['proporcao_manter']:.1%}")

proporção MANTER: 52.5%


**Decisão.** Se as classes estão próximas do equilíbrio, **não reamostramos**: usamos `class_weight="balanced"` onde houver suporte e F1-macro como métrica. Reamostrar inventaria ou apagaria palavras sem necessidade.

## 7. Divisão treino/teste
10% das **frases** (não das palavras) vão para o teste final, estratificado por domínio (geral/governo), com semente fixa. Palavras da mesma frase nunca se separam: se separassem, o modelo veria no treino a frase cujas palavras vai "prever" no teste.

In [8]:
pd.crosstab(frases["split"], frases["governo"].map({False: "geral", True: "governo"}), margins=True)

governo,geral,governo,All
split,,,
teste,7230,5502,12732
treino,65067,49521,114588
All,72297,55023,127320


In [9]:
assert palavras.groupby("frase_id")["split"].nunique().max() == 1, "frase dividida entre treino e teste!"
print("OK: nenhuma frase aparece nos dois conjuntos")
palavras.groupby("split")["rotulo"].agg(palavras="size", manter="mean").round(3)

OK: nenhuma frase aparece nos dois conjuntos

,palavras,manter
split,,
teste,160897,0.526
treino,1454368,0.525


## 8. Conferência manual da qualidade dos rótulos

A rotulagem automática erra quando o intérprete usou outro sinal para a mesma ideia (sinônimo, paráfrase): a palavra vira `DESCARTAR` sem ter sido de fato descartada. **Medir isso antes de treinar** é o que o plano exige.

**Como conferir (não exige saber Libras):** abra `data/processed/vlibrasbd/amostra_conferencia_200.csv` (100 `MANTER` e 100 `DESCARTAR`, sorteadas). Para cada linha, veja se a palavra (ou algo equivalente) está na glosa e preencha `rotulo_correto` com `1` (MANTER) ou `0` (DESCARTAR). Depois rode a célula abaixo.

In [10]:
amostra_path = config.PROCESSED_DIR / "amostra_conferencia_200.csv"
amostra = pd.read_csv(amostra_path, dtype={"rotulo_correto": "string"})
conferidas = amostra[amostra.rotulo_correto.isin(["0", "1"])].copy()
if conferidas.empty:
    print(f"Nenhuma linha conferida ainda ({len(amostra)} aguardando em {amostra_path.name}).")
else:
    conferidas["rotulo_correto"] = conferidas.rotulo_correto.astype(int)
    conferidas["errado"] = conferidas.rotulo != conferidas.rotulo_correto
    print(f"{len(conferidas)} conferidas · taxa de rótulo errado: {conferidas.errado.mean():.1%}")
    print(conferidas.groupby("rotulo")["errado"].mean().rename(index={0: "DESCARTAR", 1: "MANTER"}).round(3).to_string())

Nenhuma linha conferida ainda (200 aguardando em amostra_conferencia_200.csv).


## Saídas desta etapa

- `data/processed/vlibrasbd/palavras.parquet` (e a tabela `palavras_t1` no Postgres): uma linha por palavra, atributos + `rotulo` + `split`.
- `data/processed/vlibrasbd/frases.parquet`: frases limpas com `split`.
- `data/processed/vlibrasbd/relatorio_preparacao.json`: contagens de cada passo.

Próxima etapa: **fase 3**, baseline de Regressão Logística no MLflow, treinando só em `split == "treino"`.